# NB7 — Evidence-Weighted Graph Construction & Validation

**Purpose:** Build a protein–protein interaction (PPI) graph from STRING v12,
filtered to genes with PCBS evidence, annotate nodes with component scores,
perform PCOSKB2 and stage-overlap validation, and generate
10 explanatory graphs for downstream GNN modelling (NB8).

| Input | Source | File |
|-------|--------|------|
| PCBS scores | NB6 | `results/pcbs_scores.csv` |
| Bridge genes | NB6 | `results/pcbs_bridge_genes.csv` |
| EPS routes | NB6 | `results/eps_causal_routes.csv` (optional) |
| STRING PPI | STRING v12.0 | Downloaded (human, combined_score ≥ 700) |
| PCOSKB2 curated genes | PCOSKB2 | `PCOSKB2_ManualCurated_Genes.xlsx` (uploaded) |

**Outputs:**
- `results/ppi_edge_list.csv` — filtered edge list with combined scores
- `results/ppi_node_features.csv` — node feature matrix (P/T/C/D/K/PCBS + centrality + community)
- `results/graph_summary_stats.csv` — graph-level statistics
- `results/pcoskb2_validation.csv` — PCOSKB2 overlap analysis
- `results/stage_overlap_validation.csv` — bridge gene stage overlap analysis
- `results/NB7_Graph_results.zip` — all outputs + 10 figures

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import os, warnings, zipfile, io, requests
from collections import defaultdict, Counter
warnings.filterwarnings('ignore')

os.makedirs('results', exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 100,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

print('NB7 — Evidence-Weighted Graph Construction & Validation')
print('=' * 60)

## 1 · Load NB6 PCBS Scores

In [ ]:
from google.colab import files

print("Upload your NB6 results ZIP (NB6_PCBS_results.zip)\n")

uploaded = files.upload()

for fname, content in uploaded.items():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(content), 'r') as zf:
            zf.extractall('.')
            print(f"Extracted {fname}: {len(zf.namelist())} files")
            for name in sorted(zf.namelist()):
                print(f"  {name}")

# Load PCBS scores
pcbs = pd.read_csv('results/pcbs_scores.csv')
bridge = pd.read_csv('results/pcbs_bridge_genes.csv')

print(f'\nTotal scored genes:  {len(pcbs):,}')
print(f'Bridge genes (PCBS > 0): {len(bridge):,}')
print(f'Max PCBS: {pcbs["PCBS"].max():.4f} ({pcbs.loc[pcbs["PCBS"].idxmax(), "Gene"]})')

# Load EPS causal routes if available
eps_available = False
if os.path.exists('results/eps_causal_routes.csv'):
    eps_routes = pd.read_csv('results/eps_causal_routes.csv')
    eps_available = True
    print(f'\nEPS causal routes loaded: {len(eps_routes)} routes')
    print(f'Top route EPS: {eps_routes["EPS"].max():.4f}')
else:
    print('\nNote: eps_causal_routes.csv not found — EPS integration skipped.')
    print('Run route-level EPS cells in NB6 to generate this file.')

## 2 · Download STRING PPI Network

Download the human PPI network from STRING v12.0.  
We use the **protein.links** file (combined scores) and the **protein.aliases**
file to map STRING Ensembl protein IDs → HGNC gene symbols.

Only interactions with **combined_score ≥ 700** (high confidence) are retained.

In [ ]:
import urllib.request
import gzip

# STRING v12.0 — Homo sapiens (taxon 9606)
LINKS_URL = 'https://stringdb-downloads.org/download/protein.links.v12.0/9606.protein.links.v12.0.txt.gz'
ALIASES_URL = 'https://stringdb-downloads.org/download/protein.aliases.v12.0/9606.protein.aliases.v12.0.txt.gz'

def download_gz(url, out_path):
    """Download a .gz file and decompress it."""
    gz_path = out_path + '.gz'
    if os.path.exists(out_path):
        print(f'  Already exists: {out_path}')
        return
    print(f'  Downloading {url.split("/")[-1]} ...')
    urllib.request.urlretrieve(url, gz_path)
    with gzip.open(gz_path, 'rb') as f_in:
        with open(out_path, 'wb') as f_out:
            f_out.write(f_in.read())
    os.remove(gz_path)
    print(f'  Saved: {out_path}')

download_gz(LINKS_URL, 'string_links.txt')
download_gz(ALIASES_URL, 'string_aliases.txt')

print('\nSTRING files ready.')

## 3 · Build STRING ID → Gene Symbol Mapping

In [ ]:
# Parse aliases — keep BioMart_HUGO and Ensembl_HGNC sources for gene symbols
aliases = pd.read_csv('string_aliases.txt', sep='\t',
                       comment='#', header=0,
                       names=['string_id', 'alias', 'source'])

print(f'Total alias entries: {len(aliases):,}')
print(f'Unique STRING proteins: {aliases["string_id"].nunique():,}')

# Prefer HUGO/HGNC gene symbols
hugo_mask = aliases['source'].str.contains('BioMart_HUGO|Ensembl_HGNC', na=False)
hugo_aliases = aliases[hugo_mask].drop_duplicates(subset='string_id', keep='first')

# Build mapping dictionary
string_to_gene = dict(zip(hugo_aliases['string_id'], hugo_aliases['alias']))
print(f'Mapped STRING IDs (HUGO/HGNC): {len(string_to_gene):,}')

# Check overlap with our PCBS gene universe
pcbs_genes = set(pcbs['Gene'].values)
mapped_genes = set(string_to_gene.values())
overlap = pcbs_genes & mapped_genes
print(f'\nPCBS gene universe:     {len(pcbs_genes):,}')
print(f'STRING mapped genes:    {len(mapped_genes):,}')
print(f'Overlap (usable nodes): {len(overlap):,} ({100*len(overlap)/len(pcbs_genes):.1f}%)')

## 4 · Filter PPI Edges (High Confidence ≥ 700)

In [ ]:
# Load STRING links
print('Loading STRING links (this may take a minute)...')
links = pd.read_csv('string_links.txt', sep=' ')
print(f'Total STRING interactions (human): {len(links):,}')

# Filter high confidence
SCORE_THRESHOLD = 700
links_hc = links[links['combined_score'] >= SCORE_THRESHOLD].copy()
print(f'High confidence (≥ {SCORE_THRESHOLD}): {len(links_hc):,}')

# Map STRING IDs to gene symbols
links_hc['gene1'] = links_hc['protein1'].map(string_to_gene)
links_hc['gene2'] = links_hc['protein2'].map(string_to_gene)

# Drop unmapped
links_mapped = links_hc.dropna(subset=['gene1', 'gene2']).copy()
print(f'After gene symbol mapping: {len(links_mapped):,}')

# Remove self-loops
links_mapped = links_mapped[links_mapped['gene1'] != links_mapped['gene2']]
print(f'After removing self-loops: {len(links_mapped):,}')

# Filter to PCBS gene universe (both endpoints must be in our scored genes)
mask = (links_mapped['gene1'].isin(pcbs_genes)) & (links_mapped['gene2'].isin(pcbs_genes))
ppi_edges = links_mapped[mask][['gene1', 'gene2', 'combined_score']].copy()
ppi_edges = ppi_edges.reset_index(drop=True)

# Normalise combined_score to [0, 1]
ppi_edges['weight'] = ppi_edges['combined_score'] / 1000.0

print(f'\nFinal PPI edges (both genes in PCBS universe): {len(ppi_edges):,}')
print(f'Unique genes in PPI: {len(set(ppi_edges["gene1"]) | set(ppi_edges["gene2"])):,}')

## 5 · Construct NetworkX Graph

In [ ]:
# Build undirected graph
G = nx.Graph()

for _, row in ppi_edges.iterrows():
    G.add_edge(row['gene1'], row['gene2'],
               weight=row['weight'],
               combined_score=int(row['combined_score']))

# Add node features from PCBS
pcbs_dict = pcbs.set_index('Gene').to_dict('index')
feature_cols = ['P_g', 'T_g', 'C_g', 'D_g', 'K_g', 'PCBS', 'n_components']

for node in G.nodes():
    if node in pcbs_dict:
        for col in feature_cols:
            G.nodes[node][col] = pcbs_dict[node].get(col, 0.0)
    else:
        for col in feature_cols:
            G.nodes[node][col] = 0.0

print('Graph constructed')
print(f'  Nodes: {G.number_of_nodes():,}')
print(f'  Edges: {G.number_of_edges():,}')
print(f'  Density: {nx.density(G):.6f}')

# Connected components
components = list(nx.connected_components(G))
largest_cc = max(components, key=len)
print(f'  Connected components: {len(components)}')
print(f'  Largest component: {len(largest_cc):,} nodes '
      f'({100*len(largest_cc)/G.number_of_nodes():.1f}%)')

# Degree stats
degrees = [d for _, d in G.degree()]
print(f'\nDegree statistics:')
print(f'  Mean:   {np.mean(degrees):.1f}')
print(f'  Median: {np.median(degrees):.0f}')
print(f'  Max:    {max(degrees)} ({[n for n,d in G.degree() if d==max(degrees)][0]})')
print(f'  Min:    {min(degrees)}')

## 6 · Degree Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(degrees, bins=50, color='#1565C0', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Degree')
axes[0].set_ylabel('Number of Genes')
axes[0].set_title('Degree Distribution', fontweight='bold')
axes[0].axvline(np.mean(degrees), color='red', ls='--', label=f'Mean={np.mean(degrees):.1f}')
axes[0].legend()

# Log-log (power-law check)
from collections import Counter
deg_count = Counter(degrees)
degs = sorted(deg_count.keys())
counts = [deg_count[d] for d in degs]
axes[1].scatter(degs, counts, s=15, alpha=0.6, color='#D32F2F')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('Degree (log)')
axes[1].set_ylabel('Count (log)')
axes[1].set_title('Degree Distribution (log-log)', fontweight='bold')

plt.suptitle('PPI Network Degree Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/ppi_degree_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/ppi_degree_distribution.png')

## 7 · Hub Gene Analysis

Identify genes that are both **topologically central** (high degree) and
**biologically important** (high PCBS). These are the strongest bridge candidates.

In [ ]:
# Build node summary table
node_data = []
for node in G.nodes():
    d = G.degree(node)
    attrs = G.nodes[node]
    node_data.append({
        'Gene': node,
        'Degree': d,
        'P_g': attrs.get('P_g', 0),
        'T_g': attrs.get('T_g', 0),
        'C_g': attrs.get('C_g', 0),
        'D_g': attrs.get('D_g', 0),
        'K_g': attrs.get('K_g', 0),
        'PCBS': attrs.get('PCBS', 0),
        'n_components': int(attrs.get('n_components', 0)),
    })

node_df = pd.DataFrame(node_data)

# Normalise degree to [0, 1] for hub scoring
node_df['Degree_norm'] = node_df['Degree'] / node_df['Degree'].max()

# Hub score = PCBS × Degree_norm (genes strong in both)
node_df['Hub_score'] = node_df['PCBS'] * node_df['Degree_norm']

# Top 20 hub genes
top_hubs = node_df.nlargest(20, 'Hub_score').reset_index(drop=True)

print('Top 20 Hub Genes (PCBS × Degree)')
print('=' * 75)
print(f'{"Gene":<10} {"PCBS":>7} {"Degree":>7} {"Hub":>7} {"#Comp":>6}')
print('-' * 75)
for _, r in top_hubs.iterrows():
    print(f'{r["Gene"]:<10} {r["PCBS"]:7.4f} {int(r["Degree"]):7d} '
          f'{r["Hub_score"]:7.4f} {int(r["n_components"]):6d}')

In [ ]:
# Scatter: Degree vs PCBS
fig, ax = plt.subplots(figsize=(10, 7))

sc = ax.scatter(node_df['Degree'], node_df['PCBS'],
                c=node_df['n_components'], cmap='YlOrRd',
                s=20, alpha=0.6, edgecolors='white', linewidth=0.3)
plt.colorbar(sc, label='Evidence components')

# Label top hub genes
for _, r in top_hubs.head(10).iterrows():
    ax.annotate(r['Gene'], (r['Degree'], r['PCBS']),
                fontsize=8, fontweight='bold',
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Degree (PPI connections)')
ax.set_ylabel('PCBS Score')
ax.set_title('PPI Degree vs PCBS — Hub Gene Identification', fontweight='bold')

plt.tight_layout()
plt.savefig('results/ppi_degree_vs_pcbs.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/ppi_degree_vs_pcbs.png')

## 8 · Bridge Gene Subgraph Visualisation

Extract and visualise the subgraph of top bridge genes to show how they
are interconnected via PPI.

In [ ]:
# Top bridge genes — take top 30 by PCBS that are in the graph
graph_genes = set(G.nodes())
top_bridge = (bridge[bridge['Gene'].isin(graph_genes)]
              .nlargest(30, 'PCBS')['Gene'].tolist())

# Subgraph
sub_G = G.subgraph(top_bridge).copy()

print(f'Bridge subgraph (top 30 PCBS genes in PPI):')
print(f'  Nodes: {sub_G.number_of_nodes()}')
print(f'  Edges: {sub_G.number_of_edges()}')

if sub_G.number_of_nodes() > 0:
    fig, ax = plt.subplots(figsize=(14, 12))

    # Node sizes proportional to PCBS
    pcbs_vals = [sub_G.nodes[n].get('PCBS', 0) for n in sub_G.nodes()]
    node_sizes = [300 + 2000 * v for v in pcbs_vals]

    # Node colours by n_components
    n_comp = [sub_G.nodes[n].get('n_components', 0) for n in sub_G.nodes()]

    # Edge widths by weight
    edge_weights = [sub_G[u][v].get('weight', 0.7) for u, v in sub_G.edges()]
    edge_widths = [0.5 + 2 * w for w in edge_weights]

    pos = nx.spring_layout(sub_G, seed=42, k=2.0)

    nx.draw_networkx_edges(sub_G, pos, ax=ax, width=edge_widths,
                           alpha=0.3, edge_color='gray')
    nodes = nx.draw_networkx_nodes(sub_G, pos, ax=ax,
                                   node_size=node_sizes,
                                   node_color=n_comp, cmap='YlOrRd',
                                   edgecolors='black', linewidths=0.5)
    nx.draw_networkx_labels(sub_G, pos, ax=ax, font_size=8,
                            font_weight='bold')

    plt.colorbar(nodes, label='Evidence components', shrink=0.7)
    ax.set_title('Top 30 Bridge Genes — PPI Subgraph\n'
                 '(node size = PCBS, colour = evidence depth)',
                 fontweight='bold', fontsize=13)
    ax.axis('off')

    plt.tight_layout()
    plt.savefig('results/ppi_bridge_subgraph.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: results/ppi_bridge_subgraph.png')
else:
    print('No bridge genes found in PPI network.')

## 9 · Graph Centrality Measures

Compute standard centrality metrics on the full PPI graph to identify
topologically important genes beyond simple degree.

In [ ]:
# Use largest connected component for meaningful centrality
G_lcc = G.subgraph(largest_cc).copy()

print(f'Computing centrality on largest component ({G_lcc.number_of_nodes():,} nodes)...')

# Betweenness (approximate for large graphs)
if G_lcc.number_of_nodes() > 5000:
    betw = nx.betweenness_centrality(G_lcc, k=min(500, G_lcc.number_of_nodes()))
    print('  Betweenness: approximate (k=500 samples)')
else:
    betw = nx.betweenness_centrality(G_lcc)
    print('  Betweenness: exact')

# Closeness
close = nx.closeness_centrality(G_lcc)
print('  Closeness: done')

# PageRank
pr = nx.pagerank(G_lcc, weight='weight')
print('  PageRank: done')

# Add to node_df
node_df['Betweenness'] = node_df['Gene'].map(betw).fillna(0)
node_df['Closeness'] = node_df['Gene'].map(close).fillna(0)
node_df['PageRank'] = node_df['Gene'].map(pr).fillna(0)

# Top 10 by each centrality
for metric in ['Betweenness', 'Closeness', 'PageRank']:
    top = node_df.nlargest(10, metric)
    print(f'\nTop 10 by {metric}:')
    for _, r in top.iterrows():
        print(f'  {r["Gene"]:<10} {metric}={r[metric]:.6f}  PCBS={r["PCBS"]:.4f}  '
              f'Degree={int(r["Degree"])}')

## 10 · Centrality vs PCBS Correlation

In [ ]:
from scipy import stats

centrality_cols = ['Degree', 'Betweenness', 'Closeness', 'PageRank']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, metric in zip(axes, centrality_cols):
    active = node_df[node_df['PCBS'] > 0]
    rho, pval = stats.spearmanr(active[metric], active['PCBS'])
    ax.scatter(active[metric], active['PCBS'], s=10, alpha=0.4, color='#1565C0')
    ax.set_xlabel(metric)
    ax.set_ylabel('PCBS')
    ax.set_title(f'{metric} vs PCBS\nρ = {rho:.3f} (p = {pval:.2e})',
                 fontweight='bold', fontsize=10)

plt.suptitle('Network Centrality vs Bridge Score',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/ppi_centrality_vs_pcbs.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/ppi_centrality_vs_pcbs.png')

# Summary table
print('\nCentrality–PCBS Spearman correlations (PCBS > 0 genes):')
for metric in centrality_cols:
    rho, pval = stats.spearmanr(active[metric], active['PCBS'])
    print(f'  {metric:<14} ρ = {rho:+.4f}  p = {pval:.2e}')

## 11 · Component Evidence Heatmap

Visualise the **evidence profile** of top bridge genes across all five PCBS
components (P/T/C/D/K). Genes with broad evidence support (high across many
components) are stronger bridge candidates than single-evidence genes.

In [ ]:
# Component Evidence Heatmap — Top 30 bridge genes × 5 components
top30_ev = node_df[node_df['PCBS'] > 0].nlargest(30, 'PCBS').copy()

fig, ax = plt.subplots(figsize=(10, 12))

heat_data = top30_ev.set_index('Gene')[['P_g', 'T_g', 'C_g', 'D_g', 'K_g']]
heat_data.columns = ['P: PMOS', 'T: Transition', 'C: Cancer',
                      'D: Directional', 'K: Independent']

sns.heatmap(heat_data, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, vmin=0,
            cbar_kws={'label': 'Component Score', 'shrink': 0.6})
ax.set_title('Component Evidence Profile — Top 30 Bridge Genes\n'
             '(5 PCBS components: P, T, C, D, K)',
             fontweight='bold', fontsize=13)
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig('results/component_evidence_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/component_evidence_heatmap.png')

# Evidence depth summary
print('\nEvidence depth for top 30 bridge genes:')
for comp, label in [('P_g','PMOS'), ('T_g','Transition'), ('C_g','Cancer'),
                     ('D_g','Directional'), ('K_g','Independent')]:
    has = (top30_ev[comp] > 0).sum()
    mean_val = top30_ev[comp][top30_ev[comp] > 0].mean()
    print(f'  {label:12s}: {has:2d}/30 genes, mean score = {mean_val:.4f}')

## 12 · PCBS Score Distribution

Examine the distribution of PCBS scores across the PPI network.
A right-skewed distribution is expected — most genes have low bridge scores,
with a small number of strong bridge candidates in the tail.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of PCBS > 0
pcbs_positive = node_df[node_df['PCBS'] > 0]['PCBS']
axes[0].hist(pcbs_positive, bins=50, color='#1565C0', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('PCBS Score')
axes[0].set_ylabel('Number of Genes')
axes[0].set_title('PCBS Distribution (scored genes)', fontweight='bold')
axes[0].axvline(pcbs_positive.median(), color='red', ls='--',
                label=f'Median = {pcbs_positive.median():.4f}')
axes[0].axvline(pcbs_positive.quantile(0.9), color='orange', ls=':',
                label=f'90th pct = {pcbs_positive.quantile(0.9):.4f}')
axes[0].legend(fontsize=9)

# Boxplot by evidence depth
comp_data = node_df[node_df['PCBS'] > 0].copy()
comp_data['Evidence depth'] = comp_data['n_components'].clip(upper=5).astype(str)
order = sorted(comp_data['Evidence depth'].unique())
sns.boxplot(data=comp_data, x='Evidence depth', y='PCBS', ax=axes[1],
            order=order, palette='YlOrRd', fliersize=3)
axes[1].set_xlabel('Number of Evidence Components')
axes[1].set_ylabel('PCBS Score')
axes[1].set_title('PCBS by Evidence Depth', fontweight='bold')

plt.suptitle('Bridge Score Distribution in PPI Network',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/pcbs_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/pcbs_distribution.png')

# Statistics
print(f'\nPCBS statistics (genes in PPI with PCBS > 0):')
print(f'  Count:  {len(pcbs_positive):,}')
print(f'  Mean:   {pcbs_positive.mean():.4f}')
print(f'  Median: {pcbs_positive.median():.4f}')
print(f'  Std:    {pcbs_positive.std():.4f}')
print(f'  Skew:   {pcbs_positive.skew():.2f}')
ppi_genes_total = node_df.shape[0]
ppi_genes_scored = (node_df['PCBS'] > 0).sum()
print(f'\n  PPI nodes total:       {ppi_genes_total:,}')
print(f'  PPI nodes with PCBS>0: {ppi_genes_scored:,} ({100*ppi_genes_scored/ppi_genes_total:.1f}%)')

## 13 · PPI Edge Confidence Distribution

Examine the confidence score distribution of PPI edges. Higher STRING
combined scores indicate stronger experimental or computational evidence
for the interaction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(ppi_edges['combined_score'], bins=40, color='#43A047',
             alpha=0.8, edgecolor='white')
axes[0].set_xlabel('STRING Combined Score')
axes[0].set_ylabel('Number of Edges')
axes[0].set_title('PPI Edge Confidence Distribution', fontweight='bold')
axes[0].axvline(900, color='red', ls='--', label='Highest (≥ 900)')
axes[0].axvline(ppi_edges['combined_score'].median(), color='orange',
                ls=':', label=f'Median = {ppi_edges["combined_score"].median():.0f}')
axes[0].legend(fontsize=9)

# Bridge vs non-bridge edge comparison
bridge_set = set(bridge['Gene'].values)
is_bridge_edge = (ppi_edges['gene1'].isin(bridge_set)) | (ppi_edges['gene2'].isin(bridge_set))
both_bridge = (ppi_edges['gene1'].isin(bridge_set)) & (ppi_edges['gene2'].isin(bridge_set))

axes[1].hist(ppi_edges.loc[~is_bridge_edge, 'combined_score'],
             bins=30, alpha=0.6, color='#9E9E9E', label='Non-bridge edges', density=True)
axes[1].hist(ppi_edges.loc[both_bridge, 'combined_score'],
             bins=30, alpha=0.7, color='#D32F2F', label='Both-bridge edges', density=True)
axes[1].set_xlabel('STRING Combined Score')
axes[1].set_ylabel('Density')
axes[1].set_title('Edge Confidence: Bridge vs Non-Bridge', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('PPI Network Edge Quality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/edge_weight_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/edge_weight_distribution.png')

print(f'\nEdge statistics:')
print(f'  Total edges:          {len(ppi_edges):,}')
print(f'  Mean score:           {ppi_edges["combined_score"].mean():.0f}')
print(f'  Median score:         {ppi_edges["combined_score"].median():.0f}')
print(f'  Highest conf (≥ 900): {(ppi_edges["combined_score"] >= 900).sum():,} '
      f'({100*(ppi_edges["combined_score"] >= 900).sum()/len(ppi_edges):.1f}%)')
print(f'  Bridge–Bridge edges:  {both_bridge.sum():,}')
print(f'  Bridge-incident:      {is_bridge_edge.sum():,}')

## 14 · Network Community Structure

Detect functional modules in the PPI network using the **Louvain** community
detection algorithm. Bridge genes concentrated in specific communities
may indicate coordinated pathway involvement in the PCOS→EC transition.

In [ ]:
# Community detection (Louvain on largest connected component)
print('Detecting network communities (Louvain)...')
try:
    from networkx.algorithms.community import louvain_communities
    comms = louvain_communities(G_lcc, seed=42, weight='weight', resolution=1.0)
except (ImportError, AttributeError):
    from networkx.algorithms.community import greedy_modularity_communities
    comms = list(greedy_modularity_communities(G_lcc, weight='weight'))

comms = sorted(comms, key=len, reverse=True)
print(f'Communities found: {len(comms)}')

# Community summary
bridge_set_c = set(bridge['Gene'].values)
print(f'\n{"Comm":<6} {"Size":>6} {"Bridge":>7} {"Top gene (PCBS)":>20}')
print('-' * 50)
for i, comm in enumerate(comms[:10]):
    bridge_in = [g for g in comm if g in bridge_set_c]
    # Find top PCBS gene in this community
    top_gene = ''
    top_pcbs = 0
    for g in comm:
        gp = node_df.loc[node_df['Gene'] == g, 'PCBS']
        if len(gp) > 0 and gp.values[0] > top_pcbs:
            top_pcbs = gp.values[0]
            top_gene = g
    print(f'  {i+1:<4d} {len(comm):6d} {len(bridge_in):7d}   {top_gene} ({top_pcbs:.4f})')

# Assign community labels to nodes
node_community = {}
for i, comm in enumerate(comms):
    for node in comm:
        node_community[node] = i
node_df['Community'] = node_df['Gene'].map(node_community).fillna(-1).astype(int)

# Visualise bridge gene distribution across top communities
top_n_comm = min(8, len(comms))
comm_bridge_counts = []
comm_sizes = []
for i in range(top_n_comm):
    comm_bridge_counts.append(len([g for g in comms[i] if g in bridge_set_c]))
    comm_sizes.append(len(comms[i]))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart: community sizes with bridge gene counts
x = range(top_n_comm)
axes[0].bar(x, comm_sizes, color='#BBDEFB', edgecolor='#1565C0', label='Total genes')
axes[0].bar(x, comm_bridge_counts, color='#D32F2F', alpha=0.8, label='Bridge genes')
axes[0].set_xlabel('Community')
axes[0].set_ylabel('Number of Genes')
axes[0].set_title('Community Composition', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'C{i+1}' for i in x])
axes[0].legend()

# Network: top bridge genes coloured by community
top_bridge_comm = (node_df[(node_df['PCBS'] > 0) & (node_df['Community'] >= 0)]
                   .nlargest(40, 'PCBS')['Gene'].tolist())
sub_comm = G.subgraph(top_bridge_comm).copy()

if sub_comm.number_of_nodes() > 0:
    pos = nx.spring_layout(sub_comm, seed=42, k=2.5)
    comm_colors = [node_community.get(n, 0) for n in sub_comm.nodes()]
    pcbs_sizes = [300 + 2000 * sub_comm.nodes[n].get('PCBS', 0) for n in sub_comm.nodes()]

    edge_w = [0.5 + 2 * sub_comm[u][v].get('weight', 0.7) for u, v in sub_comm.edges()]
    nx.draw_networkx_edges(sub_comm, pos, ax=axes[1], width=edge_w,
                           alpha=0.3, edge_color='gray')
    sc = nx.draw_networkx_nodes(sub_comm, pos, ax=axes[1],
                                node_size=pcbs_sizes,
                                node_color=comm_colors, cmap='Set2',
                                edgecolors='black', linewidths=0.5, alpha=0.8)
    nx.draw_networkx_labels(sub_comm, pos, ax=axes[1], font_size=7, font_weight='bold')
    plt.colorbar(sc, ax=axes[1], label='Community', shrink=0.6)
    axes[1].set_title('Top 40 Bridge Genes by Community\n(size ∝ PCBS)',
                      fontweight='bold')
    axes[1].axis('off')

plt.suptitle('Network Community Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/network_communities.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/network_communities.png')

## 15 · PCOSKB2 Validation

Validate bridge genes against the **PCOSKB2** manually curated gene list
(Sharma et al., *Scientific Reports*, 2020; DOI: 10.1038/s41598-020-62657-w).

PCOSKB2 contains **533 experimentally validated PCOS-associated genes** curated
from 1,561 PubMed publications. Overlap of our computationally identified
bridge genes with this curated set validates the PCOS biological relevance
of the bridge gene panel.

**Test:** Fisher's exact test for enrichment of PCOSKB2 genes among bridge
genes, with the full PCBS-scored gene universe as background.

In [ ]:
# ── PCOSKB2 Validation ──────────────────────────────────────────────
from google.colab import files
from scipy.stats import fisher_exact

print('Upload PCOSKB2_ManualCurated_Genes.xlsx\n')
uploaded_kb = files.upload()

# Read PCOSKB2 gene list from uploaded Excel
import openpyxl
pcoskb2_genes = set()
for fname, content in uploaded_kb.items():
    if fname.endswith('.xlsx'):
        wb = openpyxl.load_workbook(io.BytesIO(content), read_only=True)
        ws = wb.active
        for row in ws.iter_rows(min_row=2, values_only=True):
            if row and row[1]:  # Column B = Gene Symbol
                pcoskb2_genes.add(str(row[1]).strip())
        wb.close()
        print(f'Loaded {len(pcoskb2_genes)} PCOSKB2 curated genes from {fname}')

# Define sets
bridge_genes_set = set(bridge['Gene'].values)
scored_universe  = set(pcbs['Gene'].values)
ppi_bridge       = bridge_genes_set & set(G.nodes())

# Overlaps
overlap_all    = bridge_genes_set & pcoskb2_genes
overlap_ppi    = ppi_bridge & pcoskb2_genes
pcoskb2_scored = pcoskb2_genes & scored_universe

print(f'\n{"="*60}')
print(f'PCOSKB2 Validation Summary')
print(f'{"="*60}')
print(f'PCOSKB2 curated genes:           {len(pcoskb2_genes)}')
print(f'PCOSKB2 in scored universe:      {len(pcoskb2_scored)}')
print(f'Bridge genes (PCBS > 0):         {len(bridge_genes_set):,}')
print(f'Bridge genes in PPI:             {len(ppi_bridge):,}')
print(f'\nOverlap (bridge ∩ PCOSKB2):      {len(overlap_all)}')
print(f'Overlap (PPI bridge ∩ PCOSKB2):  {len(overlap_ppi)}')

# Fisher's exact test (2×2 contingency)
# Bridge+PCOSKB2 | Bridge+NotPCOSKB2
# NotBridge+PCOSKB2 | NotBridge+NotPCOSKB2
a = len(bridge_genes_set & pcoskb2_scored)
b = len(bridge_genes_set - pcoskb2_scored)
c = len(pcoskb2_scored - bridge_genes_set)
d = len(scored_universe - bridge_genes_set - pcoskb2_scored)

print(f'\nFisher\'s exact test (bridge gene enrichment in PCOSKB2):')
print(f'  Background universe: {len(scored_universe):,} scored genes')
print(f'  Contingency table:')
print(f'                   PCOSKB2+  PCOSKB2-')
print(f'    Bridge+        {a:>7d}   {b:>7d}')
print(f'    Bridge-        {c:>7d}   {d:>7d}')

odds_ratio, pval = fisher_exact([[a, b], [c, d]], alternative='greater')
print(f'\n  Odds ratio: {odds_ratio:.3f}')
print(f'  P-value:    {pval:.2e}')
print(f'  Enrichment: {"SIGNIFICANT" if pval < 0.05 else "Not significant"} (α = 0.05)')

# Stratified PCBS analysis: overlapping vs non-overlapping bridge genes
overlap_pcbs = pcbs[pcbs['Gene'].isin(overlap_all)]['PCBS']
nonoverlap_pcbs = pcbs[(pcbs['Gene'].isin(bridge_genes_set)) &
                       (~pcbs['Gene'].isin(overlap_all))]['PCBS']

print(f'\nPCBS score comparison:')
print(f'  Overlapping (n={len(overlap_pcbs)}):      '
      f'mean={overlap_pcbs.mean():.4f}, median={overlap_pcbs.median():.4f}')
print(f'  Non-overlapping (n={len(nonoverlap_pcbs)}): '
      f'mean={nonoverlap_pcbs.mean():.4f}, median={nonoverlap_pcbs.median():.4f}')

if len(overlap_pcbs) > 0 and len(nonoverlap_pcbs) > 0:
    from scipy.stats import mannwhitneyu
    u_stat, mw_p = mannwhitneyu(overlap_pcbs, nonoverlap_pcbs, alternative='greater')
    print(f'  Mann-Whitney U test (overlap > non-overlap): p = {mw_p:.2e}')

# List overlapping genes with PCBS scores
overlap_detail = pcbs[pcbs['Gene'].isin(overlap_all)].sort_values('PCBS', ascending=False)
overlap_detail = overlap_detail[['Gene', 'P_g', 'T_g', 'C_g', 'D_g', 'K_g', 'PCBS']].copy()
print(f'\nTop overlapping genes (bridge ∩ PCOSKB2):')
print(f'{"Gene":<12} {"P_g":>6} {"T_g":>6} {"C_g":>6} {"D_g":>6} {"K_g":>6} {"PCBS":>7}')
print('-' * 55)
for _, r in overlap_detail.head(20).iterrows():
    print(f'{r["Gene"]:<12} {r["P_g"]:6.3f} {r["T_g"]:6.3f} {r["C_g"]:6.3f} '
          f'{r["D_g"]:6.3f} {r["K_g"]:6.3f} {r["PCBS"]:7.4f}')

# Save validation results
val_df = overlap_detail.copy()
val_df['In_PPI'] = val_df['Gene'].isin(set(G.nodes()))
val_df.to_csv('results/pcoskb2_validation.csv', index=False)
print(f'\nSaved: results/pcoskb2_validation.csv')

# ── Visualisation: PCOSKB2 overlap ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# LEFT: Overlap proportional bar chart
categories = ['Bridge genes\n(PCBS > 0)', 'Bridge genes\nin PPI']
total_vals = [len(bridge_genes_set), len(ppi_bridge)]
overlap_vals = [len(overlap_all), len(overlap_ppi)]
nonoverlap_vals = [t - o for t, o in zip(total_vals, overlap_vals)]

x = np.arange(len(categories))
width = 0.45
axes[0].bar(x, overlap_vals, width, label=f'In PCOSKB2 ({len(pcoskb2_genes)} curated)',
            color='#D32F2F', alpha=0.85)
axes[0].bar(x, nonoverlap_vals, width, bottom=overlap_vals,
            label='Not in PCOSKB2', color='#BBDEFB', alpha=0.85)
for i, (ov, tot) in enumerate(zip(overlap_vals, total_vals)):
    axes[0].text(i, tot + 10, f'{ov}/{tot}\n({100*ov/tot:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories)
axes[0].set_ylabel('Number of Genes')
axes[0].set_title(f'PCOSKB2 Overlap\nFisher p = {pval:.2e}, OR = {odds_ratio:.2f}',
                  fontweight='bold')
axes[0].legend(fontsize=9)

# RIGHT: PCBS distribution comparison
box_data = []
box_labels = []
if len(overlap_pcbs) > 0:
    box_data.append(overlap_pcbs.values)
    box_labels.append(f'PCOSKB2+\n(n={len(overlap_pcbs)})')
if len(nonoverlap_pcbs) > 0:
    box_data.append(nonoverlap_pcbs.values)
    box_labels.append(f'PCOSKB2−\n(n={len(nonoverlap_pcbs)})')

bp = axes[1].boxplot(box_data, labels=box_labels, patch_artist=True,
                     widths=0.5, flierprops={'markersize': 3})
colors = ['#D32F2F', '#1565C0']
for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_ylabel('PCBS Score')
axes[1].set_title('PCBS Scores: PCOSKB2-Validated vs Novel',
                  fontweight='bold')

plt.suptitle('PCOSKB2 Validation — Bridge Gene PCOS Relevance',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/pcoskb2_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/pcoskb2_validation.png')

## 16 · Stage Overlap Validation

Assess the **bridge nature** of our identified genes by examining how many
disease stages each bridge gene is differentially expressed in.

The three PCBS components map to disease stages:
- **P(g)** > 0 → DEG in PCOS endometrium (GSE48301)
- **T(g)** > 0 → DEG in endometrial hyperplasia / transition (GSE106191)
- **C(g)** > 0 → DEG in endometrial cancer (GSE17025)

Bridge genes appearing as DEGs across **2 or 3 stages** directly support the
PCOS→cancer progression hypothesis, as they are dysregulated along the
full disease continuum.

In [ ]:
# ── Stage Overlap Validation ────────────────────────────────────────
print('Stage Overlap Validation')
print('=' * 60)

# Determine stage membership for each bridge gene
# P_g > 0 → DEG in PCOS stage, T_g > 0 → Transition, C_g > 0 → Cancer
bridge_with_scores = pcbs[pcbs['PCBS'] > 0].copy()

bridge_with_scores['DEG_PCOS'] = (bridge_with_scores['P_g'] > 0).astype(int)
bridge_with_scores['DEG_Transition'] = (bridge_with_scores['T_g'] > 0).astype(int)
bridge_with_scores['DEG_Cancer'] = (bridge_with_scores['C_g'] > 0).astype(int)
bridge_with_scores['N_stages'] = (bridge_with_scores['DEG_PCOS'] +
                                   bridge_with_scores['DEG_Transition'] +
                                   bridge_with_scores['DEG_Cancer'])

# Stage overlap summary
stage_counts = bridge_with_scores['N_stages'].value_counts().sort_index()
print(f'\nBridge genes by number of disease stages (DEG):')
for n_stage, count in stage_counts.items():
    pct = 100 * count / len(bridge_with_scores)
    label = {0: 'D/K only (no stage DEG)', 1: '1 stage', 2: '2 stages', 3: '3 stages (full bridge)'}.get(n_stage, f'{n_stage} stages')
    print(f'  {label}: {count:,} genes ({pct:.1f}%)')

multi_stage = (bridge_with_scores['N_stages'] >= 2).sum()
full_bridge = (bridge_with_scores['N_stages'] == 3).sum()
print(f'\nMulti-stage bridge genes (≥2 stages): {multi_stage:,} '
      f'({100*multi_stage/len(bridge_with_scores):.1f}%)')
print(f'Full-bridge genes (all 3 stages):     {full_bridge:,} '
      f'({100*full_bridge/len(bridge_with_scores):.1f}%)')

# Per-stage breakdown
for stage, col in [('PCOS', 'DEG_PCOS'), ('Transition', 'DEG_Transition'),
                    ('Cancer', 'DEG_Cancer')]:
    n = bridge_with_scores[col].sum()
    print(f'  DEG in {stage:12s}: {n:,} genes ({100*n/len(bridge_with_scores):.1f}%)')

# Stage overlap patterns (Venn-like)
patterns = bridge_with_scores.groupby(['DEG_PCOS', 'DEG_Transition', 'DEG_Cancer']).size()
print(f'\nStage overlap patterns:')
print(f'  {"P":>3} {"T":>3} {"C":>3}  {"Count":>6}  Pattern')
print(f'  {"-"*30}')
for (p, t, c), count in patterns.items():
    labels = []
    if p: labels.append('PCOS')
    if t: labels.append('Trans')
    if c: labels.append('Cancer')
    label = '+'.join(labels) if labels else 'D/K only'
    print(f'  {p:>3} {t:>3} {c:>3}  {count:>6}  {label}')

# Top multi-stage bridge genes
multi_stage_genes = (bridge_with_scores[bridge_with_scores['N_stages'] >= 2]
                     .sort_values('PCBS', ascending=False))
print(f'\nTop 20 multi-stage bridge genes:')
print(f'{"Gene":<12} {"PCOS":>5} {"Trans":>5} {"Cancer":>6} {"Stages":>6} {"PCBS":>7}')
print('-' * 50)
for _, r in multi_stage_genes.head(20).iterrows():
    print(f'{r["Gene"]:<12} {"✓" if r["DEG_PCOS"] else "·":>5} '
          f'{"✓" if r["DEG_Transition"] else "·":>5} '
          f'{"✓" if r["DEG_Cancer"] else "·":>5} '
          f'{int(r["N_stages"]):>6} {r["PCBS"]:>7.4f}')

# Save stage overlap results
stage_save = bridge_with_scores[['Gene', 'P_g', 'T_g', 'C_g', 'D_g', 'K_g',
                                  'PCBS', 'DEG_PCOS', 'DEG_Transition',
                                  'DEG_Cancer', 'N_stages']].copy()
stage_save.sort_values('PCBS', ascending=False).to_csv(
    'results/stage_overlap_validation.csv', index=False)
print(f'\nSaved: results/stage_overlap_validation.csv')

# ── Visualisation: Stage overlap ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# LEFT: Stacked bar — bridge genes by stage count
stage_data = stage_counts.reindex([0, 1, 2, 3], fill_value=0)
colors_stage = ['#E0E0E0', '#FFB74D', '#FF7043', '#D32F2F']
labels_stage = ['D/K only', '1 stage', '2 stages', '3 stages']
bars = axes[0].bar(range(4), stage_data.values, color=colors_stage,
                   edgecolor='white', linewidth=1.5)
axes[0].set_xticks(range(4))
axes[0].set_xticklabels(labels_stage)
axes[0].set_ylabel('Number of Bridge Genes')
axes[0].set_xlabel('Disease Stages as DEG')
for bar, val in zip(bars, stage_data.values):
    if val > 0:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                     str(val), ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Bridge Gene Stage Coverage\n'
                  f'Multi-stage: {multi_stage} ({100*multi_stage/len(bridge_with_scores):.0f}%)',
                  fontweight='bold')

# RIGHT: Heatmap — top 30 bridge genes × 3 stages
top30_stage = multi_stage_genes.head(30).copy()
if len(top30_stage) > 0:
    heat_stage = top30_stage.set_index('Gene')[['DEG_PCOS', 'DEG_Transition', 'DEG_Cancer']]
    heat_stage.columns = ['PCOS\n(P_g > 0)', 'Transition\n(T_g > 0)', 'Cancer\n(C_g > 0)']

    sns.heatmap(heat_stage, annot=True, fmt='d', cmap='RdYlGn',
                linewidths=0.5, ax=axes[1], vmin=0, vmax=1,
                cbar_kws={'label': 'DEG status', 'shrink': 0.4,
                          'ticks': [0, 1]})
    axes[1].set_title('Stage DEG Profile\nTop 30 Multi-Stage Bridge Genes',
                      fontweight='bold')
    axes[1].set_ylabel('')
    axes[1].tick_params(axis='y', labelsize=8)

plt.suptitle('Stage Overlap Validation — Disease Progression Evidence',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/stage_overlap_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/stage_overlap_validation.png')

## 17 · Known Bridge Genes in Network

Validate our computationally identified bridge genes against literature-curated
PCOS–EC shared genes. Coverage of known genes in the PPI network confirms
the network captures established biology.

In [ ]:
known_bridge_genes = [
    'AKT1', 'PIK3CA', 'PTEN', 'ESR1', 'AR', 'INSR', 'IGF1R',
    'CYP19A1', 'CYP11A1', 'STAR', 'HSD3B2',
    'IL6', 'TNF', 'VEGFA', 'TP53',
    'ERBB2', 'ERBB3', 'FOXO1', 'CCND1', 'MYC',
]

print(f'{"Gene":<10} {"InPPI":>6} {"Degree":>7} {"PCBS":>7} {"Betw":>9} {"PageRank":>9}')
print('-' * 60)

in_ppi = 0
for gene in known_bridge_genes:
    in_graph = gene in graph_genes
    if in_graph:
        in_ppi += 1
        row = node_df[node_df['Gene'] == gene].iloc[0]
        print(f'{gene:<10} {"Yes":>6} {int(row["Degree"]):7d} {row["PCBS"]:7.4f} '
              f'{row["Betweenness"]:9.6f} {row["PageRank"]:9.6f}')
    else:
        pcbs_row = pcbs[pcbs['Gene'] == gene]
        pcbs_val = pcbs_row['PCBS'].values[0] if len(pcbs_row) > 0 else 0
        print(f'{gene:<10} {"No":>6} {"—":>7} {pcbs_val:7.4f} {"—":>9} {"—":>9}')

print(f'\nKnown genes in PPI network: {in_ppi}/{len(known_bridge_genes)}')

## 18 · Save Results

In [ ]:
# Edge list
ppi_edges[['gene1', 'gene2', 'combined_score', 'weight']].to_csv(
    'results/ppi_edge_list.csv', index=False)
print(f'Saved: results/ppi_edge_list.csv  ({len(ppi_edges):,} edges)')

# Node features (now includes Community)
node_save_cols = ['Gene', 'Degree', 'P_g', 'T_g', 'C_g', 'D_g', 'K_g',
                  'PCBS', 'n_components', 'Betweenness', 'Closeness',
                  'PageRank', 'Community']
node_df.sort_values('PCBS', ascending=False)[node_save_cols].to_csv(
    'results/ppi_node_features.csv', index=False)
print(f'Saved: results/ppi_node_features.csv  ({len(node_df):,} nodes)')

# Graph summary statistics
components = list(nx.connected_components(G))
largest_cc = max(components, key=len)
degrees = [d for _, d in G.degree()]

summary = {
    'Metric': [
        'Total nodes', 'Total edges', 'Density',
        'Connected components', 'Largest CC nodes', 'Largest CC fraction',
        'Mean degree', 'Median degree', 'Max degree',
        'Bridge genes in PPI', 'STRING score threshold',
        'Communities (Louvain)', 'Largest community',
        'PCOSKB2 overlap (bridge)', 'PCOSKB2 Fisher p-value',
        'Multi-stage bridge genes',
    ],
    'Value': [
        G.number_of_nodes(), G.number_of_edges(), f'{nx.density(G):.6f}',
        len(components), len(largest_cc),
        f'{100*len(largest_cc)/G.number_of_nodes():.1f}%',
        f'{np.mean(degrees):.1f}', f'{np.median(degrees):.0f}', max(degrees),
        len(bridge[bridge['Gene'].isin(graph_genes)]),
        SCORE_THRESHOLD,
        node_df['Community'].nunique(),
        node_df.groupby('Community').size().max() if 'Community' in node_df.columns else 'N/A',
        f'{len(overlap_all)} / {len(bridge_genes_set)}',
        f'{pval:.2e}',
        f'{multi_stage} / {len(bridge_with_scores)}',
    ]
}
summary_df = pd.DataFrame(summary)
summary_df.to_csv('results/graph_summary_stats.csv', index=False)
print(f'Saved: results/graph_summary_stats.csv')

print(f'\n{"="*60}')
print(f'GRAPH CONSTRUCTION & VALIDATION COMPLETE')
print(f'{"="*60}')
print(f'Nodes:  {G.number_of_nodes():,}')
print(f'Edges:  {G.number_of_edges():,}')
print(f'Bridge genes in PPI: {len(bridge[bridge["Gene"].isin(graph_genes)]):,}')
print(f'Communities: {node_df["Community"].nunique()}')
print(f'PCOSKB2 overlap: {len(overlap_all)} bridge genes validated')
print(f'Multi-stage bridge genes: {multi_stage}')

## Summary

NB7 constructed a high-confidence PPI graph from STRING v12 (≥ 700),
filtered to the PCBS gene universe and annotated with five evidence components.
Two validation analyses were performed: PCOSKB2 overlap to confirm PCOS
biological relevance, and stage overlap to validate bridge gene dysregulation
across the PCOS→cancer disease continuum.

**10 Explanatory Figures:**

| # | Figure | Description |
|---|--------|-------------|
| 1 | Degree distribution | Histogram + log-log power-law check |
| 2 | Degree vs PCBS | Hub gene identification scatter |
| 3 | Bridge gene subgraph | Top 30 bridge genes PPI network |
| 4 | Centrality vs PCBS | Spearman correlation (4 metrics) |
| 5 | Component evidence heatmap | P/T/C/D/K profile for top 30 genes |
| 6 | PCBS distribution | Score distribution + evidence depth boxplot |
| 7 | Edge weight distribution | STRING confidence + bridge vs non-bridge |
| 8 | Network communities | Louvain community structure + bridge enrichment |
| 9 | PCOSKB2 validation | Overlap bar chart + PCBS comparison boxplot |
| 10 | Stage overlap validation | Stage coverage bar + multi-stage heatmap |

| Output | Description |
|--------|-------------|
| `ppi_edge_list.csv` | Filtered PPI edges with combined scores |
| `ppi_node_features.csv` | Node feature matrix (P/T/C/D/K/PCBS + centrality + community) |
| `graph_summary_stats.csv` | Graph-level statistics |
| `pcoskb2_validation.csv` | PCOSKB2 overlap analysis |
| `stage_overlap_validation.csv` | Bridge gene stage overlap analysis |

**Next:** NB8 trains the Evidence-Gated GNN on this graph and benchmarks
against PPR / GCN / GraphSAGE / GAT / GIN baselines.

In [ ]:
zip_path = 'results/NB7_Graph_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    result_files = [
        'results/ppi_edge_list.csv',
        'results/ppi_node_features.csv',
        'results/graph_summary_stats.csv',
        'results/pcoskb2_validation.csv',
        'results/stage_overlap_validation.csv',
        'results/ppi_degree_distribution.png',
        'results/ppi_degree_vs_pcbs.png',
        'results/ppi_bridge_subgraph.png',
        'results/ppi_centrality_vs_pcbs.png',
        'results/component_evidence_heatmap.png',
        'results/pcbs_distribution.png',
        'results/edge_weight_distribution.png',
        'results/network_communities.png',
        'results/pcoskb2_validation.png',
        'results/stage_overlap_validation.png',
    ]
    for f in result_files:
        if os.path.exists(f):
            zf.write(f)
            print(f'  Added: {f}')
        else:
            print(f'  Skipped (not found): {f}')

print(f'\nResults ZIP: {zip_path}')

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not running in Colab — download manually.')